# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [ ]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Periksa nama file di folder data/raw/."
    )

df = pl.read_parquet(DATA_PATH)

print(f"File: {DATA_PATH.name}")
print(f"Ukuran awal: {df.height:,} baris x {df.width} kolom")
print("Kolom:", df.columns)

df.head()

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [ ]:
connection = duckdb.connect()
connection.register("dataset", df.to_arrow())

profil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
).pl()

profil

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [ ]:
jumlah_baris = df.height

if jumlah_baris == 0:
    raise ValueError("Dataset kosong.")

null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100)
        .round(2)
        .alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

null_summary

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [ ]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
kolom_outlier = [
    kolom for kolom in kolom_numerik
    if not any(kata in kolom.lower()
               for kata in ["id", "date", "tanggal", "year", "tahun"])
]

df_outlier_handled = df_missing_handled.clone()

if kolom_outlier:
    KOLOM_NUMERIK = kolom_outlier[0]

    nilai = df_missing_handled[KOLOM_NUMERIK].drop_nulls()

    if len(nilai) > 0:
        q1 = nilai.quantile(0.25)
        q3 = nilai.quantile(0.75)
        iqr = q3 - q1

        batas_bawah = q1 - 1.5 * iqr
        batas_atas = q3 + 1.5 * iqr

        jumlah_outlier = (
            (nilai < batas_bawah) | (nilai > batas_atas)
        ).sum()

        print("Kolom yang diperiksa:", KOLOM_NUMERIK)
        print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
        print(f"Batas bawah: {batas_bawah}")
        print(f"Batas atas: {batas_atas}")
        print(f"Jumlah outlier: {jumlah_outlier:,}")

        df_outlier_handled = df_missing_handled.with_columns(
            pl.col(KOLOM_NUMERIK).clip(
                lower_bound=batas_bawah,
                upper_bound=batas_atas
            )
        )
    else:
        print("Kolom numerik tidak memiliki nilai valid.")
else:
    KOLOM_NUMERIK = None
    batas_bawah = None
    batas_atas = None
    print("Tidak ada kolom numerik yang cocok untuk IQR.")
    print("Outlier numerik dilewati; data teks tidak diubah.")

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [ ]:
nama_kolom = {kolom.lower(): kolom for kolom in df_outlier_handled.columns}

KOLOM_URL = next(
    (asli for nama, asli in nama_kolom.items()
     if nama in ["url", "link", "article_url", "news_url"]),
    None
)

KOLOM_JUDUL = next(
    (asli for nama, asli in nama_kolom.items()
     if any(kata in nama for kata in ["title", "judul", "headline"])),
    None
)

KOLOM_ISI = next(
    (asli for nama, asli in nama_kolom.items()
     if any(kata in nama for kata in ["content", "article", "text", "isi", "berita"])),
    None
)

if KOLOM_URL:
    KOLOM_KUNCI = [KOLOM_URL]
elif KOLOM_JUDUL and KOLOM_ISI and KOLOM_JUDUL != KOLOM_ISI:
    KOLOM_KUNCI = [KOLOM_JUDUL, KOLOM_ISI]
elif KOLOM_ISI:
    KOLOM_KUNCI = [KOLOM_ISI]
elif KOLOM_JUDUL:
    KOLOM_KUNCI = [KOLOM_JUDUL]
else:
    KOLOM_KUNCI = []

total_baris = df_outlier_handled.height

if KOLOM_KUNCI:
    df_dedup = df_outlier_handled.unique(
        subset=KOLOM_KUNCI,
        keep="first",
        maintain_order=True
    )

    baris_unik = df_dedup.height

    print("Kolom kunci:", KOLOM_KUNCI)
    print(f"Total baris: {total_baris:,}")
    print(f"Baris sesudah deduplikasi: {baris_unik:,}")
    print(f"Duplikat dihapus: {total_baris - baris_unik:,}")
else:
    df_dedup = df_outlier_handled.clone()

    print("Kolom kunci berita tidak ditemukan.")
    print("Deduplikasi dilewati agar data tidak terhapus keliru.")

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [ ]:
connection.register("dataset_bersih", df_dedup.to_arrow())

if KOLOM_NUMERIK is not None:
    validasi = connection.execute("""
        SELECT
            COUNT(*) AS total_baris,
            COUNT(*) FILTER (
                WHERE jumlah_null > 0
            ) AS pemeriksaan
        FROM (
            SELECT *
            FROM dataset_bersih
        )
    """).pl()

    # Validasi null per kolom dan jumlah baris
    null_akhir = df_dedup.null_count().sum_horizontal().sum()

    print("Total baris:", f"{df_dedup.height:,}")
    print("Total nilai null:", f"{null_akhir:,}")
    print("Kolom numerik yang diperiksa:", KOLOM_NUMERIK)
else:
    null_akhir = df_dedup.null_count().sum_horizontal().sum()

    print("Total baris:", f"{df_dedup.height:,}")
    print("Total nilai null:", f"{null_akhir:,}")
    print("Pemeriksaan IQR dilewati karena tidak ada kolom numerik.")

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {df.height:,}")
print(f"Baris sesudah cleaning: {df_dedup.height:,}")
print(f"Perubahan jumlah baris: {df.height - df_dedup.height:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.